# AI Engineering Interview Prep
## Section: LLMOps & Production AI

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


In [ ]:
# ── Optional: Install dependencies for runnable code cells ────────────────
# Uncomment and run this cell first if you want to execute the code examples

# !pip install sentence-transformers faiss-cpu numpy pydantic langchain
# !pip install langchain-community langchain-google-genai
# !pip install groq openai anthropic
# !pip install ragas datasets evaluate

print("📚 AI Engineering Interview Prep Notebook")
print("✅ Ready to study! All 651+ questions are answered below.")
print()
print("Sections:")
sections = [
    "1. LLM Fundamentals (63 Qs)",
    "2. Prompt Engineering (30 Qs)",
    "3. RAG (37 Qs)",
    "4. AI Agents (37 Qs)",
    "5. Fine-Tuning (25 Qs)",
    "6. Vector Databases (22 Qs)",
    "7. AI System Design (42 Qs)",
    "8. LLMOps (41 Qs)",
    "9. Evaluation (29 Qs)",
    "10. AI Safety & Ethics (38 Qs)",
    "11. Multimodal AI (26 Qs)",
    "12. AI Infrastructure (27 Qs)",
    "13. Coding Practicals (22 Qs)",
    "14. Behavioral Questions (22 Qs)",
    "15. LangChain (10 Qs)",
    "16. LangGraph (10 Qs)",
    "17. Python (30 Qs)",
    "18. FastAPI (30 Qs)",
    "19. Resume-Based Questions (10 Qs)",
    "21. System Design & Scenario-Based (100 Qs)",
    "20. Quick Reference Cheat Sheet",
]
for s in sections:
    print(f"  📌 Section {s}")


---
## ⚙️ Section 8 — LLMOps and Production AI

### Q1. How does Prompt Caching work?
**A:** Prompt caching stores the KV cache of a fixed prefix (system prompt + documents) so it doesn't need to be recomputed for every request.

**How it works:**
1. On the first request, the LLM computes attention over the entire prompt (expensive prefill)
2. The resulting KV cache for the fixed prefix is stored in GPU memory (or saved to disk)
3. On subsequent requests, the fixed prefix KV cache is loaded → only the new query tokens need attention computation

**Supported by:** Anthropic Claude (cache_control), Google Gemini (cachedContent API), some vLLM deployments.

**Savings:** If your system prompt is 10K tokens and you get 1000 queries/day: without caching = 10M tokens/day billed for the prefix alone. With caching = first request + negligible subsequent prefix cost.

🏷️ *Nyaya-Pro uses Gemini's CachedContent API — the legal context (Constitution + BNS overview = ~32K tokens) is cached for 1 hour. ~75% token cost reduction.*


### Q2. Explain the AI product lifecycle from ideation to production.
**A:**
1. **Problem definition:** Is AI actually needed? Define success metrics (accuracy, latency, cost, user satisfaction).
2. **Data audit:** Do we have training/evaluation data? Can we collect it?
3. **Prototype (≈ 1 week):** Quick proof-of-concept. Often just prompt engineering with a powerful model.
4. **Evaluation framework:** Build golden test set, define metrics BEFORE iterating.
5. **Iteration:** Improve prompt, try RAG, try fine-tuning — always measure against test set.
6. **Production hardening:** Add retry logic, fallbacks, rate limiting, monitoring, cost tracking.
7. **Shadow deployment:** Run new system in parallel with old, compare outputs — no user impact.
8. **Canary release:** Route 5% traffic to new system; monitor metrics for 24-48hrs.
9. **Full rollout:** Gradually increase traffic.
10. **Continuous improvement:** Monitor, collect feedback, retrain/re-prompt, repeat.


### Q3. What is LLMOps, and how does it differ from traditional MLOps?
**A:**
| | Traditional MLOps | LLMOps |
|--|------------------|--------|
| Model artefact | Trained model binary | Pre-trained LLM (weights) |
| Primary control | Model retraining | Prompt engineering, RAG, fine-tuning |
| Versioning | Model versions | Prompt versions + model versions |
| Evaluation | Accuracy, F1 on test set | LLM-as-judge, human eval, faithfulness |
| Inference | Standard REST | Streaming, token-level metrics |
| Deployment | Container + REST API | vLLM, continuous batching, KV cache |
| Cost model | Compute hours | Per-token billing |
| Key failure mode | Distribution shift | Hallucination, prompt injection |
| Monitoring | Prediction accuracy | Toxicity, faithfulness, latency, cost |

LLMOps builds on MLOps but adds prompt management, LLM-specific evaluation, and token-level cost tracking as first-class concerns.


### Q4. How do you serve LLMs in production?
**A:**
**Self-hosted (open models):**
- **vLLM**: Best throughput, PagedAttention, continuous batching — production standard
- **TensorRT-LLM (NVIDIA)**: Maximum GPU utilisation via optimised CUDA kernels
- **Ollama**: Easy local/dev deployment
- **llama.cpp**: CPU serving, great for edge

**Managed APIs:**
- OpenAI, Anthropic, Google Gemini — zero infra overhead, pay per token
- Groq — fastest inference (LPDDR-based silicon)
- AWS Bedrock, Azure OpenAI — enterprise compliance

**Serving considerations:**
- Load balancer → inference server pool (Kubernetes)
- Model warmup: pre-load model weights before serving traffic
- Rolling deployment: update one pod at a time (zero downtime)
- GPU affinity: pin model to specific GPUs to avoid loading overhead

🏷️ *JobPilot AI: Groq API for production (speed), Ollama local for dev (free, private). FastAPI backend handles routing.*


### Q5. What is model quantisation?
**A:** Quantisation reduces the numerical precision of model weights from FP32 (32 bits) to lower bit-widths, shrinking model size and speeding up inference.

| Format | Bits | Size vs FP32 | Quality | Use case |
|--------|------|-------------|---------|----------|
| FP32 | 32 | 1× | Baseline | Training |
| BF16 | 16 | 2× | ≈FP32 | GPU inference |
| FP16 | 16 | 2× | ≈FP32 | GPU inference |
| INT8 | 8 | 4× | ~1% loss | Deployment |
| INT4 (GPTQ/AWQ) | 4 | 8× | ~2-3% loss | Consumer GPU |
| 2-bit | 2 | 16× | Notable loss | Extreme edge |

**Methods:** GPTQ (post-training, fast), AWQ (activation-aware, better quality than GPTQ), bitsandbytes (QLoRA training).

**Practical:** BF16 for GPU serving (default), INT4 for consumer GPUs and edge devices.


### Q6. How do you monitor LLM applications in production?
**A:**
**Infrastructure metrics:**
- GPU utilisation, memory, temperature
- API latency (TTFT, inter-token latency, total latency)
- Error rates, timeout rates, retry rates
- Cost per request (tokens_in + tokens_out × price)

**Quality metrics (the hard part):**
- Faithfulness score (RAG: answer grounded in context?)
- Hallucination rate (spot checks with LLM-as-judge)
- Output format compliance rate (is JSON always valid?)
- Toxicity / safety filter trigger rate

**Business metrics:**
- User satisfaction (thumbs up/down, CSAT)
- Task completion rate
- Escalation rate (% handed to humans)

**Tooling:** LangSmith, Arize AI, Weights & Biases, Helicone, custom Grafana dashboards.

🏷️ *Nyaya-Pro: Prometheus for infrastructure metrics, custom middleware logs every request (latency, tokens, cost) to PostgreSQL, weekly manual faithfulness audit on a sample.*


### Q7. What is LLM observability?
**A:** Observability = the ability to understand what your LLM application is doing from the outside, through its outputs and logs, without re-instrumenting code.

**Three pillars for LLMs:**

1. **Traces:** Full execution trace — each LLM call, tool call, retrieval step — with inputs, outputs, latency, token counts. Tools: LangSmith traces, Arize Phoenix.

2. **Metrics:** Aggregated statistics — P50/P95/P99 latency, token consumption, error rates, cost/request. Tools: Prometheus + Grafana.

3. **Logs:** Structured logs of every significant event — prompt used, model version, response, user feedback. Store in Elasticsearch or BigQuery.

**What to instrument:**
- Every LLM call: prompt (sanitised), response, model, tokens, latency, cost
- Every retrieval: query, retrieved chunks, similarity scores
- Every tool call: tool name, arguments, result, latency
- User feedback: thumbs up/down, explicit ratings

🏷️ *Nyaya-Pro: custom middleware logs all three to structured JSON in PostgreSQL. Weekly review of traces for quality issues.*


### Q8. What are guardrails for LLMs, and how do you implement them?
**A:** Guardrails are validation layers that check LLM inputs and outputs to prevent harmful, off-topic, or malformed responses.

**Types:**
1. **Input guardrails:** Check user input before sending to LLM
   - Toxicity/hate speech classifier
   - PII detector (regex + NER)
   - Off-topic classifier
   - Prompt injection detector

2. **Output guardrails:** Check LLM response before returning to user
   - Topic compliance: is this response about the allowed domain?
   - Format validation: is the JSON valid and matches schema?
   - PII leakage: did LLM accidentally include PII?
   - Hallucination check: is the response faithful to the context?
   - Toxicity/harmful content

**Tools:** NVIDIA NeMo Guardrails, Guardrails AI (`guardrails-ai`), LLM Guard, custom rule-based validators.

**Production pattern:** Input guard → LLM → Output guard. If output fails → retry with corrective instructions or return fallback message.


### Q9. How do you implement content filtering for AI outputs?
**A:**
**Multi-layer approach:**

1. **Rule-based (fast, zero cost):**
   - Regex for PII patterns (email, phone, SSN, credit card)
   - Keyword blocklist for obvious violations
   - Output format validation

2. **ML classifier (fast, good quality):**
   - Fine-tuned BERT/RoBERTa for: toxicity, hate speech, NSFW, off-topic
   - Models: Jigsaw Perspective API, OpenAI Moderation API, custom classifier
   - Run asynchronously if latency-sensitive

3. **LLM-based (accurate, expensive):**
   - Ask a second LLM: "Is this response harmful/inappropriate/off-topic? yes/no and why"
   - Only for borderline cases identified by ML classifier

4. **Human review:**
   - Sample 1% of all outputs for human quality review
   - All flagged outputs go to human review queue


### Q10. How do you estimate the cost of running an AI-powered feature in production?
**A:**
**Cost formula:**
```
Daily cost = Daily_requests × (avg_input_tokens + avg_output_tokens) × token_price
           + embedding_cost (input_tokens × embedding_token_price)
           + infrastructure_cost (GPU hours, API gateway, DB, etc.)
```

**Estimation process:**
1. Measure avg prompt length (input tokens) with your real prompts
2. Measure avg response length (output tokens) empirically
3. Get token prices from provider pricing page
4. Estimate QPS from product requirements

**Example (Nyaya-Pro):**
- System prompt: ~2K tokens, user query: ~100 tokens, retrieved context: ~1.5K tokens → 3.6K input tokens
- Response: ~500 output tokens
- Gemini Flash pricing: $0.075/1M input + $0.30/1M output
- 1000 queries/day: $0.27/day input + $0.15/day output = ~$0.42/day = ~$13/month

**Cost optimisation:** Smaller input (better chunking/retrieval), shorter output (concise instructions), cheaper model, caching.


### Q11. How do you optimise LLM inference costs in production?
**A:**
1. **Model right-sizing** — use the cheapest model that meets quality threshold (GPT-4o-mini instead of GPT-4o for simple tasks)
2. **Prompt caching** — cache long system prompts; save 60-90% on prefix tokens
3. **Semantic caching** — cache responses for semantically similar queries; 20-40% cache hit rate is achievable
4. **Input compression** — shorten prompts (remove verbose examples, use concise instructions)
5. **Output limiting** — set max_tokens conservatively; models often over-generate
6. **Batch processing** — batch multiple requests together (only for async, non-real-time tasks)
7. **Model routing** — classify query complexity → route simple to cheap models, complex to expensive
8. **Self-hosted open models** — for high volume, self-hosting Llama 3 on reserved GPUs can be 10-100× cheaper than API pricing
9. **Speculative decoding** — reduce generation latency (and per-hour GPU cost) with draft model


### Q12. How do you implement A/B testing for LLM systems?
**A:**
**What to A/B test:** prompt versions, models, RAG configurations, output formats.

**Process:**
1. Define success metrics: user satisfaction, task completion rate, output quality score, cost
2. Build evaluation set (or use production traffic)
3. Implement traffic splitting: route X% to variant A, Y% to variant B
4. Ensure both versions are logged identically with `experiment_id` tag
5. Run for sufficient time (statistical significance): usually 100-1000+ samples
6. Analyse: compare distributions of metrics; compute p-value
7. Document decision and learnings

**Challenges specific to LLMs:**
- Output is stochastic — same input can produce different outputs; need enough samples
- "Quality" is hard to measure automatically — add LLM-as-judge scoring or user thumbs up/down
- Cost differences between variants must be factored in (quality AND cost)

**Tools:** Posthog, LaunchDarkly for traffic splitting; custom analysis in Python/SQL.


### Q13. What is CI/CD for AI applications, and how does it differ from traditional CI/CD?
**A:**
**Traditional CI/CD:** code change → unit tests → build → deploy.

**AI CI/CD adds:**
1. **Prompt evaluation:** run prompt changes against golden test set; fail build if quality drops
2. **Model version pinning:** ensure model version is locked; alert when provider updates the model
3. **Regression testing:** compare new version vs current production on a standard test set
4. **Cost regression:** alert if new prompt/model increases average cost per request significantly
5. **Safety testing:** automated red-team tests; fail if safety regressions detected
6. **Shadow deployment:** test new version on shadow traffic before canary

**Pipeline:**
```
Code/Prompt change → Unit tests → LLM eval on golden set
→ Cost estimate → Safety tests → Review → Canary deploy (5%)
→ Monitor 24hrs → Full deploy
```

**Tools:** LangSmith evaluations, pytest for unit tests, GitHub Actions for CI, Argo Rollouts for canary.


### Q14. How do you version and manage prompts in production?
**A:**
**Prompts are code — treat them like code:**

1. **Git versioning** — prompts stored in code (Python constants or JSON config files). Every change goes through code review.
2. **Semantic versioning** — prompts have versions: `legal_qa_v1.2.3`. Patch: typo fix. Minor: behavioural tweak. Major: format change.
3. **Prompt registry** — centralised store (LangSmith, PromptLayer, or internal DB) with version history, metadata, and evaluation scores per version.
4. **Immutable production prompts** — never edit a deployed prompt directly. Create a new version, evaluate, deploy.
5. **Rollback capability** — ability to instantly rollback to previous prompt version if production issues detected.
6. **Change documentation** — for every prompt change: what changed, why, evaluation results before and after.
7. **Environment separation** — `dev`, `staging`, `prod` use explicit prompt versions; no "latest" in production.


### Q15. What is model versioning, and how do you handle model rollbacks?
**A:**
**Why model versioning is critical:** LLM providers silently update their models. `gpt-4` today ≠ `gpt-4` in 3 months. Quality/behaviour can change without notice.

**Model pinning:**
- Always use specific model versions: `gpt-4o-2024-08-06`, not `gpt-4o`
- Subscribe to provider changelogs
- Automated smoke tests triggered when provider releases a new model version

**Rollback strategy:**
1. New model version → runs in shadow mode for 24hrs
2. Compare quality metrics vs current production
3. If regression detected: immediately pin back to previous version
4. Root cause: re-evaluate test cases that failed; adjust prompts or update guardrails
5. Retry with adjusted configuration

**Database schema for tracking:** `{request_id, model_version, prompt_version, tokens, latency, quality_score}` — allows filtering all requests by model version to diagnose regressions.


### Q16. How do you implement rate limiting and throttling for LLM APIs?
**A:**
**Two layers:**

**Provider-side limits** (your LLM API):
- LLM providers impose TPM (tokens per minute) and RPM (requests per minute) limits
- Implement exponential backoff on 429 errors: retry after 1s, 2s, 4s, 8s
- Pre-emptive throttling: track your current usage; slow down before hitting limits
- Multiple API keys for different services (distribute load)

**Client-facing limits** (your API):
- Per-user rate limits: 60 req/min, 100K tokens/day (Redis sliding window counter)
- Per-tenant limits: prevent one customer from consuming all capacity
- Priority queues: premium users get lower latency
- Request queuing: excess requests queue rather than fail immediately

**Implementation:**
```python
# Redis sliding window
pipe = redis.pipeline()
now = time.time()
key = f"rate:{user_id}:tokens"
pipe.zremrangebyscore(key, 0, now - 60)  # remove old entries
pipe.zadd(key, {request_id: now})
pipe.zcard(key)
result = pipe.execute()
if result[2] > RPM_LIMIT: raise RateLimitError
```


### Q17. How do you handle model updates and migrations without downtime?
**A:**
**Blue-green deployment for LLMs:**
1. **Blue** = current production model/prompt
2. **Green** = new model/prompt version (deployed to a new endpoint)
3. Test Green with shadow traffic or on a test set
4. Gradually shift traffic: 5% → 25% → 50% → 100%
5. Monitor quality metrics at each step; rollback if regressions
6. Once 100% on Green, decommission Blue

**Challenges specific to LLMs:**
- Model state: unlike stateless APIs, LLMs have conversation history — manage context format changes during migration
- Prompt compatibility: new model versions may require prompt adjustments
- Evaluation lag: quality regressions in LLMs may take hours of production traffic to detect

**Key tool:** Feature flags (LaunchDarkly) allow instant rollback without redeployment — just flip the flag from Green back to Blue.


### Q18. What is the role of feature flags in AI deployments?
**A:** Feature flags decouple code deployment from feature activation — enabling instant on/off control without redeploys.

**Use cases in AI systems:**
1. **Model rollout:** deploy new model code but only activate for 5% of users via flag
2. **Prompt changes:** new prompt code deployed; activate flag to switch production
3. **Feature kill switch:** if production model starts misbehaving, disable the AI feature instantly without a hotfix deploy
4. **A/B testing:** flags route different user segments to different model/prompt variants
5. **Gradual rollout:** 1% → 5% → 25% → 100% with automated quality gates between steps

**Implementation:** LaunchDarkly, Posthog, or simple DB-backed feature flag service. Check flag value at the start of each request.

🏷️ *At Yotta, I would implement this as config-driven routing: `if feature_flags.get("use_gemini_flash"): model = gemini_flash else: model = gemini_pro`*


### Q19. How do you implement logging and tracing for LLM applications?
**A:**
**Structured logging (every LLM call):**
```json
{
  "timestamp": "2024-06-15T10:30:00Z",
  "request_id": "uuid",
  "user_id": "user123",
  "session_id": "sess456",
  "model": "gemini-1.5-flash",
  "prompt_version": "legal_qa_v2.1",
  "input_tokens": 3600,
  "output_tokens": 450,
  "latency_ms": 780,
  "cost_usd": 0.00042,
  "retrieved_chunks": ["BNS_302_1", "BNS_302_2"],
  "quality_score": null
}
```

**Distributed tracing:** Use OpenTelemetry to trace across multiple services. A single user request may involve: API gateway → RAG service → LLM service → response formatter. Each span tagged with trace_id.

**Tools:** LangSmith (purpose-built for LLM traces), Jaeger + OpenTelemetry (general distributed tracing), ELK stack (log aggregation).

🏷️ *FastAPI middleware in all my projects logs structured JSON to stdout → collected by Loki → queried in Grafana.*


### Q20. How do you handle PII and sensitive data in LLM inputs and outputs?
**A:**
**Input PII handling:**
1. **Detect before sending to LLM:** use presidio (Microsoft) or spaCy NER to detect names, emails, phone numbers, SSNs, credit cards
2. **Pseudonymise:** replace PII with placeholders before LLM processing: "John Smith" → "[PERSON_1]", "555-1234" → "[PHONE_1]"
3. **Restore after:** map placeholders back to real values in the response if needed

**Output PII handling:**
4. **Scan LLM output** for PII patterns before returning to user — LLMs can memorise or infer PII
5. **Redact or flag:** mask unexpected PII in output

**Data minimisation:**
- Don't send more context than needed
- Log sanitised prompts (PII removed) for debugging

**Regulatory compliance:**
- GDPR: data subject rights, data minimisation, lawful basis for processing
- HIPAA: PHI must be de-identified before sending to third-party LLMs
- Enterprise: use on-premise LLMs or private cloud deployment for sensitive data


### Q21. What is a gateway pattern for LLM API management?
**A:** An AI gateway is a centralised proxy between your application services and LLM providers — analogous to an API gateway but optimised for LLM traffic.

**Responsibilities:**
- Authentication: validate API keys, JWT tokens
- Routing: which model/provider handles this request (based on request type, cost, availability)
- Rate limiting: enforce per-user and per-tenant limits
- Cost tracking: meter every token consumed per user/team
- Caching: semantic and exact-match cache
- Fallbacks: automatic failover to secondary providers
- Logging: structured logs for all LLM traffic
- Content filtering: input/output guardrails
- Load balancing: distribute across multiple API keys

**Benefits:** Centralised control, consistent observability, ability to swap LLM providers without changing application code, cost visibility.

**Open-source:** LiteLLM Proxy (supports 100+ LLM providers behind one API), Portkey, HelixAI.


### Q22. How does Token Streaming work?
**A:** Instead of waiting for the full response before returning it, streaming sends tokens to the client as they are generated.

**Without streaming:** User sends request → waits 5 seconds → receives full response at once. Perceived as slow even if total time is the same.

**With streaming:** User sends request → first token appears in 200ms → tokens stream at 50-100 tokens/sec → response completes after 5s. Perceived as much faster.

**Implementation (Server-Sent Events):**
```python
@app.get("/stream")
async def stream_response(query: str):
    async def generate():
        async for chunk in llm.stream(query):
            yield f"data: {chunk.text}

"
    return StreamingResponse(generate(), media_type="text/event-stream")
```

**Client:** `EventSource` API in browser reads the SSE stream and appends tokens to the UI as they arrive.

🏷️ *All three of my projects use streaming — FastAPI StreamingResponse → SSE → React frontend appends tokens in real-time.*


### Q23. How do you implement streaming responses for real-time AI applications?
**A:** (See Q22 for basics.) Production streaming implementation:

**Server side:**
```python
async def stream_llm(prompt: str, request: Request):
    async for chunk in llm.astream(prompt):
        if await request.is_disconnected():
            break  # client disconnected, stop generating
        yield chunk.text
```

**Key considerations:**
1. **Connection drops:** check `request.is_disconnected()` periodically; stop generation if client gone (saves tokens/cost)
2. **Error handling:** if mid-stream error occurs, send an error token and close stream gracefully
3. **Buffering:** avoid buffering entire response before sending (defeats the purpose)
4. **WebSockets:** alternative to SSE for bidirectional streaming (chat apps)
5. **Load balancers:** configure timeout and buffering settings for streaming-compatible behaviour (AWS ALB: disable proxy buffering)
6. **Token-by-token vs chunk streaming:** some providers send tokens individually, others send in chunks of N tokens


### Q24. What are the key SLAs and metrics for production AI systems?
**A:**
**Latency SLAs:**
- TTFT (Time to First Token): < 500ms for conversational apps
- Total response time: < 5s for standard queries
- P95 latency: < 3× median (avoid tail latency killing UX)

**Throughput:**
- Tokens/second: benchmark your serving setup
- Concurrent requests: how many can be served simultaneously?

**Quality SLAs:**
- Faithfulness: > 95% (for RAG systems)
- Format compliance: > 99.9%
- Safety filter pass rate: > 99.99%

**Availability:**
- Uptime: 99.9% (8.7hr downtime/year) — standard
- 99.99% for critical systems (52min downtime/year)

**Cost SLAs:**
- Cost per query: defined budget; alert if exceeded
- Monthly LLM spend: budget with auto-throttle at 90%

**Tooling:** Prometheus + Grafana for all metrics; PagerDuty for SLA breach alerts.


### Q25. Cloud vs on-device model deployment for AI applications.
**A:**
| | Cloud (API) | Self-hosted Cloud | On-device |
|--|-------------|------------------|---------|
| **Privacy** | Data leaves device | Data in your cloud | Data stays on device |
| **Latency** | Network latency | Low (in-VPC) | Zero network latency |
| **Cost** | Per token (scales with usage) | Fixed GPU cost | Zero marginal cost |
| **Model quality** | Best (frontier models) | Good (open models) | Limited (SLMs) |
| **Reliability** | Provider uptime | You manage | Works offline |
| **Scale** | Infinite (provider handles) | You manage | Limited by device |
| **Best for** | Most apps | Privacy-sensitive enterprise | Mobile, offline, privacy-critical |

**Decision framework:**
- Need GPT-4 quality → Cloud API
- Data privacy + high volume → Self-hosted open model
- Mobile app / offline → On-device SLM
- Mixed → hybrid (on-device for private data, cloud for complex queries)


### Q26. How do you implement fallback strategies when the primary model is unavailable?
**A:**
```python
class LLMWithFallback:
    def __init__(self):
        self.providers = [
            {"name": "groq", "client": groq_client, "model": "llama-3.3-70b"},
            {"name": "openai", "client": openai_client, "model": "gpt-4o-mini"},
            {"name": "gemini", "client": gemini_client, "model": "gemini-flash"},
        ]
    
    async def generate(self, prompt: str) -> str:
        for provider in self.providers:
            try:
                return await provider["client"].generate(
                    prompt, model=provider["model"], timeout=5.0
                )
            except (TimeoutError, RateLimitError, APIError) as e:
                log.warning(f"{provider['name']} failed: {e}")
                continue
        return get_cached_fallback(prompt)  # last resort
```

**Circuit breaker:** If provider fails 5× in 60s, stop trying for 60s (avoid cascading delays).

**Monitoring:** Track which provider is serving which % of traffic; alert if fallback usage > 5%.

🏷️ *JobPilot AI: Groq primary → OpenAI fallback → Ollama local emergency. Groq handles 99% of traffic; others are safety nets.*


### Q27. How do you implement structured output from LLMs reliably in production?
**A:**
**Reliability spectrum (worst to best):**

1. **Plain prompt instruction only** (~80% reliability): "Respond with JSON: {schema}". Fragile.

2. **Few-shot examples** (~90%): Include 2-3 example input→JSON pairs. Better.

3. **JSON mode / response_format** (~98%): OpenAI: `response_format={"type": "json_object"}`. Forces valid JSON.

4. **Schema-constrained JSON mode** (~99.5%): Gemini: `response_schema=MyPydanticModel`. Forces JSON matching exact schema.

5. **Grammar-constrained decoding** (~99.99%): llama.cpp with formal grammar; Outlines library. Constrains token sampling to only valid JSON tokens at each step.

6. **Validation + auto-retry** (production safety net):
```python
for attempt in range(3):
    response = llm.generate(prompt)
    try:
        return MyModel.model_validate_json(response)
    except ValidationError as e:
        prompt += f"
Previous response was invalid: {e}. Fix and retry."
```

🏷️ *Nyaya-Pro: Gemini JSON mode + Pydantic validation + 3× retry. Zero format failures in production since implementing this.*


### Q28. How do you handle long contexts efficiently in production?
**A:**
1. **Prefix caching (prompt caching):** Cache the KV cache of the fixed system prompt + static context. Only dynamic parts re-computed each request.
2. **Context compression:** Use an LLM to compress the conversation history into a shorter summary after every N turns.
3. **Sliding window:** Keep only the most recent K tokens; discard oldest turns.
4. **RAG instead of long context:** Don't send 100K tokens; retrieve the 2K most relevant tokens.
5. **Chunked processing:** For very long documents, process in chunks and aggregate results (map-reduce).
6. **Speculative prefill:** For known long prefixes (system prompts), pre-compute and cache KV before user request arrives.
7. **Flash Attention / Flash Attention 2:** Enables efficient attention over long sequences on GPU (O(N) memory instead of O(N²)).
8. **Choose a long-context model:** Gemini 1.5 Pro (1M tokens), Claude 3 (200K) for genuinely long documents.


### Q29. What is semantic routing, and how do you implement it in a multi-model system?
**A:** Semantic routing = classify the incoming query by intent/domain, then route to the most appropriate model or subsystem.

**Implementation:**
```python
ROUTE_DESCRIPTIONS = {
    "legal_query": "Questions about Indian law, BNS, Constitution, legal rights",
    "general_chat": "General conversation, greetings, off-topic",
    "document_analysis": "User wants to analyse or summarise a specific document",
}

def route_query(query: str) -> str:
    # Embed query + route descriptions, pick closest
    query_emb = embed(query)
    route_embs = {k: embed(v) for k, v in ROUTE_DESCRIPTIONS.items()}
    return max(route_embs, key=lambda k: cosine_sim(query_emb, route_embs[k]))
```

**Then route to appropriate model:**
- `legal_query` → Gemini Pro + Nyaya RAG pipeline
- `general_chat` → Gemini Flash (cheaper)
- `document_analysis` → Claude 3 Sonnet (better at long-context analysis)

🏷️ *Nyaya-Pro uses LLM-based semantic routing to classify queries into Criminal/Constitutional/Civil before routing to the correct FAISS index subset.*


### Q30. How do you manage secrets and API keys securely in LLM applications?
**A:**
1. **Never hardcode API keys** in source code. Use environment variables.
2. **Secret manager:** AWS Secrets Manager, HashiCorp Vault, Azure Key Vault — store all API keys here; application fetches at runtime.
3. **Environment separation:** Different API keys for dev, staging, production. Dev keys have rate limits and no production data access.
4. **Rotation policy:** Rotate API keys every 90 days. Automate rotation with Secrets Manager.
5. **Least privilege:** Each service uses its own API key with only the permissions it needs.
6. **Audit logging:** Log every secret access (who, when, which secret).
7. **Key scanning in CI:** Use trufflehog or gitleaks to scan git commits for accidentally committed secrets.
8. **Key vaulting in Kubernetes:** Use Kubernetes Secrets + secrets-store-csi-driver to mount secrets from Vault into pods.

🏷️ *Nyaya-Pro: All API keys in `.env` file (gitignored) locally, AWS Secrets Manager in production. FastAPI reads via `python-dotenv` locally and directly from environment in AWS.*


### Q31. Your LLM API has latency spikes during peak hours. How do you stabilise it?
**A:**
1. **Identify the bottleneck:** Is it the LLM API (provider-side) or your infrastructure (DB, retrieval)?
2. **Add request queuing:** Don't hit the LLM API directly. Queue requests (Redis/SQS) and process at a controlled rate.
3. **Multiple API keys:** Distribute requests across multiple keys to avoid per-key rate limits.
4. **Fallback providers:** Route to a secondary LLM provider when primary is slow (circuit breaker pattern).
5. **Prioritise requests:** Premium users get queue priority; background tasks can wait.
6. **Pre-compute during off-peak:** For predictable queries (e.g., daily reports), generate them at 3am.
7. **Upgrade LLM provider tier:** Enterprise tiers have higher rate limits and priority queue.
8. **Async + streaming:** Start streaming first token immediately; users perceive less latency.


### Q32. Your LLM costs are too high in production. How do you reduce costs without degrading quality?
**A:** In order of impact:
1. **Model downgrade** — test GPT-4o-mini or Gemini Flash for your use case; 90% of tasks don't need frontier quality (biggest impact)
2. **Prompt caching** — if your system prompt is long (>1K tokens), prompt caching saves 60-90% on repeated requests
3. **Semantic caching** — 20-40% of queries are semantically similar to past queries; serve cached responses
4. **Shorter prompts** — audit prompt length; remove verbose instructions; use concise imperatives
5. **Reduce output tokens** — `max_tokens` + explicit length instructions ("answer in 2 sentences")
6. **Better retrieval** — retrieve fewer, better chunks (reranking); less context = fewer input tokens
7. **Model routing** — classify queries: simple → cheap model, complex → expensive model
8. **Batch async requests** — non-real-time tasks batched at off-peak hours at lower priority

🏷️ *Nyaya-Pro cost cut: switched partial traffic to Gemini Flash (-70% cost) + prompt caching (-75% on prefix tokens) = 5× total cost reduction.*


### Q33. Your application is hitting LLM provider rate limits during peak hours. How do you handle it?
**A:**
1. **Implement retry with exponential backoff:**
```python
for attempt in range(5):
    try:
        return llm.generate(prompt)
    except RateLimitError:
        wait = 2 ** attempt + random.uniform(0, 1)
        await asyncio.sleep(wait)
```
2. **Multiple API keys** — rotate through several keys; different keys have independent rate limits
3. **Request queue** — buffer requests in Redis; rate-limit consumption from queue to stay under limits
4. **Fallback provider** — if primary rate-limited, immediately route to secondary (OpenAI → Groq → Gemini)
5. **Priority queuing** — interactive user requests get priority; background jobs wait
6. **Predictive throttling** — track your TPM usage; slow down proactively before hitting limits
7. **Upgrade tier** — enterprise plans have higher rate limits
8. **Distribute load** — spread workload to multiple regions (different provider rate limit quotas per region)


### Q34. Your application depends on one LLM provider. How do you switch providers without downtime?
**A:**
**Preventing lock-in (do this now):**
1. **LiteLLM abstraction:** Use LiteLLM which provides a unified API for 100+ providers. Switching = changing one config value.
2. **Provider-agnostic prompt format:** Avoid provider-specific features (OpenAI-specific parameters, etc.)
3. **Modular provider clients:** Wrap provider clients behind your own interface; swap implementations

**When you need to switch urgently:**
1. Configure the new provider client alongside the old one (feature flag controls which is active)
2. Test new provider on shadow traffic for 24hrs
3. Flip the feature flag: all traffic now goes to new provider
4. Zero downtime — no redeployment needed

**Evaluation before switch:** Run your evaluation test set on both providers; confirm new provider quality is acceptable before switching production traffic.

🏷️ *All my projects use LangChain's model abstraction — switching from OpenAI to Gemini was literally one line of code: `ChatGoogleGenerativeAI` instead of `ChatOpenAI`.*


### Q35. Your AI system handles 100 req/sec but crashes at 5000. How do you scale?
**A:**
**Diagnosis:**
- Is the bottleneck the LLM API rate limit? (check provider usage dashboard)
- Is it your application server (CPU/memory)? (Kubernetes metrics)
- Is it the vector DB? (query latency under load)
- Is it your database? (connection pool exhaustion)

**Solutions:**
1. **Horizontal scaling** — add more application server pods (Kubernetes HPA)
2. **Async processing** — use async/await everywhere; don't block on I/O
3. **Request queuing** — buffer with Kafka/SQS; process at max sustainable rate
4. **Connection pooling** — pool DB and vector DB connections (don't create per-request)
5. **Caching** — semantic cache reduces actual LLM calls by 30-40%
6. **Multiple LLM API keys** — each key has its own rate limit; distribute load
7. **Batch processing** — combine multiple requests into one LLM call where possible
8. **CDN** — cache static content, reduce load on application servers


### Q36. A traffic spike brings down your AI system. How do you handle peak traffic?
**A:**
**Reactive (after the fact):**
1. Immediately scale out application pods (Kubernetes)
2. Enable request queuing — buffer excess requests instead of dropping them
3. Shed load gracefully: return 503 with retry-after header for lowest-priority requests
4. Increase LLM provider capacity (upgrade tier, add API keys)

**Proactive (design for this):**
1. **Auto-scaling:** HPA on CPU + queue depth; scale before hitting limits
2. **Load shedding policy:** define which request types are dropped first (background jobs first, interactive requests last)
3. **Bulkhead pattern:** allocate dedicated capacity pools for different request types; one pool crashing doesn't affect others
4. **Cached content first:** serve cached responses when freshly generated would require overloaded resources
5. **Load testing:** regular load tests to find capacity limits before users find them (k6, Locust)
6. **Circuit breakers:** prevent cascade failures when one component is overwhelmed


### Q37. One LLM provider outage took down your entire system. How do you eliminate single points of failure?
**A:**
**Multi-provider architecture:**
```
LLM Gateway
├── Primary: Groq (Llama 3.3-70b) — fast
├── Secondary: OpenAI (gpt-4o-mini) — reliable
├── Tertiary: Gemini Flash — low cost
└── Emergency: Local Ollama (always available)
```

**Circuit breaker per provider:** Stop calling a provider immediately when it errors; fail fast to fallback.

**Async health checks:** Ping each provider every 30s; pre-warm secondary before primary fails.

**Data layer resilience:**
- Vector DB: read replicas across AZs; writes to primary, reads can go to replicas
- Session state: Redis with AOF persistence + replica for durability

**Testing:** Chaos engineering — regularly kill the primary provider in staging to ensure fallback works correctly before you need it in production.


### Q38. Your multi-LLM pipeline fails when one model in the chain breaks. How do you handle orchestration failure?
**A:**
**Step-level error handling:**
1. **Retry with backoff** at the failing step (transient failures: network, timeout)
2. **Step-level fallback** — if the primary model for step 3 fails, use a secondary model for step 3 only
3. **Partial result return** — steps 1-2 succeeded; return their output with a note that step 3 failed
4. **Checkpoint / resume** — save state after each successful step; on failure, resume from last checkpoint (LangGraph checkpointing)
5. **Compensation actions** — if step 4 fails after step 3 made irreversible changes, trigger rollback (saga pattern)

**Pipeline design for resilience:**
- Each step is stateless and idempotent (safe to retry)
- Explicit output validation between steps (catch data errors early)
- Dead letter queue for permanently failed pipelines (send to human review)
- Alerting: notify on-call when pipeline failure rate > threshold

🏷️ *CrewAI in JobPilot AI: each agent has max_retries=2 and falls back to a simpler prompt on consecutive failures.*


### Q39. Your AI pipeline has zero visibility into which step is failing. How do you add observability?
**A:**
**Immediate fix:**
1. Add structured logging to every step with a `trace_id` (UUID per request, passed through all steps)
2. Log: step_name, input_tokens, output_preview, latency, success/error

**Production observability stack:**
```python
with tracer.start_as_current_span("rag_pipeline") as root_span:
    root_span.set_attribute("user_id", user_id)
    
    with tracer.start_as_current_span("retrieval"):
        chunks = retrieve(query)  # auto-logged
    
    with tracer.start_as_current_span("llm_generation"):
        response = llm.generate(chunks, query)  # auto-logged
```

**Tools:**
- **LangSmith:** native LangChain tracing — zero extra code if using LangChain
- **OpenTelemetry + Jaeger:** general distributed tracing
- **Helicone:** LLM-specific proxy logging

🏷️ *I added OpenTelemetry to Nyaya-Pro's FastAPI backend — every request now has a full trace from HTTP request to vector retrieval to LLM response, visible in Grafana Tempo.*


### Q40-38 Combined: Quantisation loss and graceful degradation.

### Q40. You quantized your LLM but accuracy dropped significantly. How do you minimise quantisation loss?
**A:**
1. **Try AWQ over GPTQ** — AWQ (Activation-aware Weight Quantisation) is generally higher quality than GPTQ at the same bit-width
2. **Increase bit-width** — switch from INT4 to INT8; higher precision = better quality
3. **Mixed-precision quantisation** — keep sensitive layers (first/last) in FP16; quantise middle layers more aggressively
4. **Quantisation-aware training (QAT)** — fine-tune the model with simulated quantisation noise; model learns to compensate
5. **Per-channel quantisation** — quantise each output channel independently (vs per-tensor); better quality
6. **KV cache quantisation separately** — quantise weights but keep KV cache in FP16 for quality

---

### Q41 (Formerly 38). One failing AI component can take down your entire platform. How do you design graceful degradation?
**A:**
1. **Bulkhead pattern** — isolate components in separate thread/process pools; one overloaded component can't starve others
2. **Timeout + fallback** — every LLM call has a strict timeout; fallback to cached/rule-based on timeout
3. **Circuit breaker** — stop calling a failing component to prevent cascade failures; return cached data
4. **Static fallbacks** — pre-built responses for critical queries served when AI is unavailable
5. **Staged degradation** — degrade gracefully: AI fails → rule-based → human → "try later"
6. **Separate health** — vector DB health check independent of LLM health; if vector DB is down, answer from LLM knowledge only
7. **Observability** — know immediately when any component degrades; PagerDuty alert within 1 minute

Design mantra: *assume every component will fail; design for failure as the normal case.*
